# Alberta Tight-Hour Warning Model

## Goal
Build a reproducible historical warning system for hours when **AIL/load is high and wind generation is low**.

The notebook will:
1. validate and explore the hourly CSV;
2. define a physically interpretable tight-hour target;
3. benchmark against **same hour last week**;
4. engineer leakage-safe calendar, lag, ramp, and rolling features;
5. use chronological validation and an untouched future test set;
6. compare logistic regression, random forest, and histogram gradient boosting;
7. report precision, recall, F1, PR-AUC, confusion matrices, and warning counts;
8. change the definition of *tight* once and quantify the resulting warning volume;
9. examine pool prices afterward as supporting context.

`pool_price_cad_per_mwh` is **not** used to define the target. Only lagged price is optionally available to the model, so the task remains a high-demand/low-wind warning problem rather than a price-labeling exercise.

> This is a historical pattern detector, not a full Alberta dispatch or reliability model.

## 1. Experimental design

Time-series data must not be randomly shuffled. The final **20% of time** is held out as an unseen test period. All target cutoffs are calculated from the earlier training period only.

### Initial definition

An hour is tight when

\[
AIL_t \ge Q_{0.80}(AIL)
\quad\text{and}\quad
Wind_t \le Q_{0.20}(Wind).
\]

These cutoffs are simple, explainable, and easy to change. Because the two conditions must occur together, the positive class can be substantially rarer than 20%.

Feature engineering also obeys causality: rolling values are shifted by one hour and no current-hour AIL, wind, or price is supplied to the classifier.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

## 2. Load and validate the CSV

Put the full CSV beside this notebook, or change `CSV_PATH`. Required columns are `timestamp`, `pool_price_cad_per_mwh`, `ail_mw`, and `wind_mw`.

In [ ]:
CSV_PATH = Path("alberta_power.csv")  # change if necessary

required = {"timestamp", "pool_price_cad_per_mwh", "ail_mw", "wind_mw"}
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"{CSV_PATH} not found. Put the CSV beside this notebook or update CSV_PATH."
    )

df = pd.read_csv(CSV_PATH)
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
for c in ["pool_price_cad_per_mwh", "ail_mw", "wind_mw"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = (df.dropna(subset=["timestamp"])
        .drop_duplicates("timestamp", keep="last")
        .sort_values("timestamp")
        .reset_index(drop=True))

print(f"Rows: {len(df):,}")
print(f"Time range: {df.timestamp.min()} -> {df.timestamp.max()}")
print("\nMissing values:")
print(df[list(required)].isna().sum())

intervals = df.timestamp.diff().dt.total_seconds().div(3600)
print("\nMost common timestamp intervals (hours):")
print(intervals.value_counts().head())
df.head(10)

In [ ]:
df[["pool_price_cad_per_mwh", "ail_mw", "wind_mw"]].describe(
    percentiles=[.05, .20, .50, .80, .95]
).T

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df.timestamp, df.ail_mw, linewidth=.7)
axes[0].set_ylabel("AIL (MW)")
axes[0].set_title("Hourly Alberta electricity data")
axes[1].plot(df.timestamp, df.wind_mw, linewidth=.7)
axes[1].set_ylabel("Wind (MW)")
axes[2].plot(df.timestamp, df.pool_price_cad_per_mwh, linewidth=.7)
axes[2].set_ylabel("Pool price\n(CAD/MWh)")
axes[2].set_xlabel("Time")
plt.tight_layout()
plt.show()

## 3. Hold out the future and define tight hours

The split happens before calculating the 80th/20th percentile cutoffs. That prevents the future test distribution from influencing the definition used during training.

In [ ]:
TEST_FRACTION = 0.20
split_idx = int(len(df) * (1 - TEST_FRACTION))
split_time = df.iloc[split_idx].timestamp

train_raw = df.iloc[:split_idx].copy()

LOAD_Q, WIND_Q = .80, .20
load_threshold = train_raw.ail_mw.quantile(LOAD_Q)
wind_threshold = train_raw.wind_mw.quantile(WIND_Q)

df["tight"] = (
    (df.ail_mw >= load_threshold) &
    (df.wind_mw <= wind_threshold)
).astype(int)

print(f"Training load cutoff Q80: {load_threshold:,.1f} MW")
print(f"Training wind cutoff Q20: {wind_threshold:,.1f} MW")
print(f"Train/test boundary: {split_time}")
print(f"Training tight rate: {df.iloc[:split_idx].tight.mean():.2%}")
print(f"Test tight rate: {df.iloc[split_idx:].tight.mean():.2%}")

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    df.iloc[:split_idx].wind_mw,
    df.iloc[:split_idx].ail_mw,
    c=df.iloc[:split_idx].tight,
    alpha=.35, s=12
)
plt.axhline(load_threshold, linestyle="--", label=f"Load cutoff {load_threshold:,.0f} MW")
plt.axvline(wind_threshold, linestyle="--", label=f"Wind cutoff {wind_threshold:,.0f} MW")
plt.xlabel("Wind generation (MW)")
plt.ylabel("AIL / load (MW)")
plt.title("Training data and tight-hour region")
plt.legend()
plt.show()

## 4. Feature engineering and weekly baseline

The benchmark prediction for hour \(t\) is simply the observed tight/not-tight status at \(t-168\) hours.

The ML features include:
- hour/day/week seasonality;
- AIL and wind lags from 1 hour through 1 week;
- lagged pool price;
- trailing means and volatility;
- one-hour load and wind ramps.

No contemporaneous target variables are predictors.

In [ ]:
def add_features(x):
    x = x.copy()
    ts = x.timestamp
    x["hour"] = ts.dt.hour
    x["dow"] = ts.dt.dayofweek
    x["month"] = ts.dt.month
    x["is_weekend"] = (x.dow >= 5).astype(int)

    x["hour_sin"] = np.sin(2*np.pi*x.hour/24)
    x["hour_cos"] = np.cos(2*np.pi*x.hour/24)
    x["dow_sin"] = np.sin(2*np.pi*x.dow/7)
    x["dow_cos"] = np.cos(2*np.pi*x.dow/7)

    for lag in [1,2,3,6,12,24,48,168]:
        x[f"ail_lag_{lag}"] = x.ail_mw.shift(lag)
        x[f"wind_lag_{lag}"] = x.wind_mw.shift(lag)

    for lag in [1,24,168]:
        x[f"price_lag_{lag}"] = x.pool_price_cad_per_mwh.shift(lag)

    prior_ail = x.ail_mw.shift(1)
    prior_wind = x.wind_mw.shift(1)
    for w in [6,24,72,168]:
        x[f"ail_mean_{w}h"] = prior_ail.rolling(w).mean()
        x[f"ail_std_{w}h"] = prior_ail.rolling(w).std()
        x[f"wind_mean_{w}h"] = prior_wind.rolling(w).mean()
        x[f"wind_std_{w}h"] = prior_wind.rolling(w).std()

    x["ail_change_1h"] = x.ail_mw.shift(1) - x.ail_mw.shift(2)
    x["wind_change_1h"] = x.wind_mw.shift(1) - x.wind_mw.shift(2)
    x["baseline_last_week"] = x.tight.shift(168)
    return x

feature_cols = [
    "hour_sin","hour_cos","dow_sin","dow_cos","month","is_weekend",
    *[f"ail_lag_{i}" for i in [1,2,3,6,12,24,48,168]],
    *[f"wind_lag_{i}" for i in [1,2,3,6,12,24,48,168]],
    *[f"price_lag_{i}" for i in [1,24,168]],
    *[f"ail_mean_{i}h" for i in [6,24,72,168]],
    *[f"ail_std_{i}h" for i in [6,24,72,168]],
    *[f"wind_mean_{i}h" for i in [6,24,72,168]],
    *[f"wind_std_{i}h" for i in [6,24,72,168]],
    "ail_change_1h","wind_change_1h"
]

model_df = add_features(df).dropna(
    subset=feature_cols + ["baseline_last_week"]
).reset_index(drop=True)

train = model_df[model_df.timestamp < split_time].copy()
test = model_df[model_df.timestamp >= split_time].copy()

X_train, y_train = train[feature_cols], train.tight.astype(int)
X_test, y_test = test[feature_cols], test.tight.astype(int)

print(f"Features: {len(feature_cols)}")
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")

## 5. Metrics

Because tight hours are uncommon, accuracy can look excellent even for a useless model. We instead emphasize **precision**, **recall**, **F1**, and **PR-AUC (average precision)**. We also count warnings, because false alarms have an operational cost.

The probability cutoff for each ML model is selected using only chronological cross-validation predictions inside the training period.

In [ ]:
def metrics(name, y, pred, score=None):
    return {
        "model": name,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, score) if score is not None else np.nan,
        "warnings_sent": int(np.sum(pred)),
        "actual_tight_hours": int(np.sum(y))
    }

baseline_pred = test.baseline_last_week.astype(int).to_numpy()
baseline_result = metrics("Same hour last week", y_test, baseline_pred)
pd.DataFrame([baseline_result])

## 6. Real-world machine-learning models

Three common tabular classifiers are compared:

- **Logistic regression:** simple, stable linear benchmark.
- **Random forest:** bagged nonlinear trees that capture interactions.
- **Histogram gradient boosting:** boosted decision trees optimized for tabular data.

The goal is a defensible comparison, not model complexity for its own sake.

In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
        ))
    ]),
    "Random forest": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=4,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    "Histogram gradient boosting": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            learning_rate=.05, max_iter=300, max_leaf_nodes=31,
            l2_regularization=1.0, random_state=RANDOM_STATE
        ))
    ])
}

In [ ]:
def oof_probabilities(model, X, y, n_splits=5):
    splitter = TimeSeriesSplit(n_splits=n_splits)
    p = np.full(len(X), np.nan)
    for tr, va in splitter.split(X):
        m = clone(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        p[va] = m.predict_proba(X.iloc[va])[:, 1]
    return p

def f1_optimal_threshold(y, p):
    valid = ~np.isnan(p)
    yy, pp = np.asarray(y)[valid], p[valid]
    precision, recall, thresholds = precision_recall_curve(yy, pp)
    f1 = 2*precision[:-1]*recall[:-1] / (precision[:-1]+recall[:-1]+1e-12)
    i = int(np.nanargmax(f1))
    return float(thresholds[i]), float(f1[i])

thresholds = {}
for name, model in models.items():
    p = oof_probabilities(model, X_train, y_train)
    thresholds[name], cv_f1 = f1_optimal_threshold(y_train, p)
    print(f"{name:30s} threshold={thresholds[name]:.3f}; CV F1={cv_f1:.3f}")

In [ ]:
results = [baseline_result]
predictions = {"Same hour last week": baseline_pred}
probabilities, fitted_models = {}, {}

for name, template in models.items():
    m = clone(template).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:,1]
    pred = (p >= thresholds[name]).astype(int)

    fitted_models[name] = m
    probabilities[name] = p
    predictions[name] = pred
    results.append(metrics(name, y_test, pred, p))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
results_df

In [ ]:
baseline_f1 = baseline_result["f1"]
best_ml = results_df[results_df.model != "Same hour last week"].iloc[0]
best_name = best_ml.model

print(f"Weekly baseline F1: {baseline_f1:.3f}")
print(f"Best ML model: {best_name} (F1={best_ml.f1:.3f})")
print(
    "ML beats the weekly baseline on held-out F1."
    if best_ml.f1 > baseline_f1
    else "The tested ML models do not beat the weekly baseline on held-out F1."
)

In [ ]:
for name in ["Same hour last week", best_name]:
    cm = confusion_matrix(y_test, predictions[name])
    fig, ax = plt.subplots(figsize=(5,4))
    ax.imshow(cm)
    ax.set_title(f"Confusion matrix: {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0,1], ["Not tight","Tight"])
    ax.set_yticks([0,1], ["Not tight","Tight"])
    for (i,j), value in np.ndenumerate(cm):
        ax.text(j, i, str(value), ha="center", va="center")
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
for name, p in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, p)
    ap = average_precision_score(y_test, p)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
plt.axhline(y_test.mean(), linestyle="--", label=f"No-skill rate={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Held-out precision-recall curves")
plt.legend()
plt.show()

## 7. Model interpretation

Permutation importance measures how much held-out average precision falls when a feature is shuffled. It is model-agnostic and useful for checking whether the warning model relies on sensible historical signals.

Importance is **predictive, not causal**.

In [ ]:
best_model = fitted_models[best_name]
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="average_precision",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
importance = (pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False).head(15))
importance

In [ ]:
p = importance.sort_values("importance_mean")
plt.figure(figsize=(9,7))
plt.barh(p.feature, p.importance_mean, xerr=p.importance_std)
plt.xlabel("Decrease in average precision after shuffling")
plt.title(f"Permutation importance: {best_name}")
plt.tight_layout()
plt.show()

## 8. Warning timeline

The following view shows predicted probability and actual tight hours in the recent part of the unseen test interval.

In [ ]:
timeline = test[["timestamp","tight","ail_mw","wind_mw","pool_price_cad_per_mwh"]].copy()
timeline["warning"] = predictions[best_name]
timeline["probability"] = probabilities[best_name]

view = timeline.tail(min(len(timeline), 24*60))
plt.figure(figsize=(15,5))
plt.plot(view.timestamp, view.probability, label="Predicted probability")
plt.axhline(thresholds[best_name], linestyle="--", label=f"Warning threshold={thresholds[best_name]:.2f}")
actual = view[view.tight == 1]
plt.scatter(actual.timestamp, actual.probability, marker="x", s=35, label="Actual tight")
plt.ylabel("Probability")
plt.xlabel("Time")
plt.title(f"{best_name}: recent held-out warning history")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Change “tight” once and count warnings

Now make the definition stricter:

\[
AIL_t \ge Q_{0.85}(AIL)
\quad\text{and}\quad
Wind_t \le Q_{0.15}(Wind).
\]

The new percentiles still come only from the original training interval. The winning model family is retrained for this revised target, its probability threshold is selected from training-only time-series validation, and warnings are counted over the **same held-out test interval**.

In [ ]:
ALT_LOAD_Q, ALT_WIND_Q = .85, .15
alt_load_threshold = train_raw.ail_mw.quantile(ALT_LOAD_Q)
alt_wind_threshold = train_raw.wind_mw.quantile(ALT_WIND_Q)

alt = model_df.copy()
alt["tight_alt"] = (
    (alt.ail_mw >= alt_load_threshold) &
    (alt.wind_mw <= alt_wind_threshold)
).astype(int)

train_alt = alt[alt.timestamp < split_time]
test_alt = alt[alt.timestamp >= split_time]
X_train_alt, y_train_alt = train_alt[feature_cols], train_alt.tight_alt
X_test_alt, y_test_alt = test_alt[feature_cols], test_alt.tight_alt

template = models[best_name]
alt_oof = oof_probabilities(template, X_train_alt, y_train_alt)
alt_threshold, alt_cv_f1 = f1_optimal_threshold(y_train_alt, alt_oof)

alt_model = clone(template).fit(X_train_alt, y_train_alt)
alt_prob = alt_model.predict_proba(X_test_alt)[:,1]
alt_pred = (alt_prob >= alt_threshold).astype(int)

original_warnings = int(predictions[best_name].sum())
changed_warnings = int(alt_pred.sum())

sensitivity = pd.DataFrame([
    {
        "definition": "Original Q80 load + Q20 wind",
        "load_cutoff_mw": load_threshold,
        "wind_cutoff_mw": wind_threshold,
        "actual_tight_test": int(y_test.sum()),
        "warnings_sent_test": original_warnings,
        "precision": precision_score(y_test, predictions[best_name], zero_division=0),
        "recall": recall_score(y_test, predictions[best_name], zero_division=0),
        "f1": f1_score(y_test, predictions[best_name], zero_division=0)
    },
    {
        "definition": "Changed Q85 load + Q15 wind",
        "load_cutoff_mw": alt_load_threshold,
        "wind_cutoff_mw": alt_wind_threshold,
        "actual_tight_test": int(y_test_alt.sum()),
        "warnings_sent_test": changed_warnings,
        "precision": precision_score(y_test_alt, alt_pred, zero_division=0),
        "recall": recall_score(y_test_alt, alt_pred, zero_division=0),
        "f1": f1_score(y_test_alt, alt_pred, zero_division=0)
    }
])
sensitivity

In [ ]:
difference = changed_warnings - original_warnings
pct = 100*difference/original_warnings if original_warnings else np.nan
print(f"Original warnings: {original_warnings:,}")
print(f"Changed-definition warnings: {changed_warnings:,}")
print(f"Difference: {difference:+,}")
if np.isfinite(pct):
    print(f"Percent change: {pct:+.1f}%")

plt.figure(figsize=(8,5))
plt.bar(["Original definition","Changed definition"], [original_warnings, changed_warnings])
plt.ylabel("Warnings in held-out test period")
plt.title("Sensitivity of warning volume to the tight-hour definition")
plt.show()

## 10. Did tight hours actually coincide with higher prices?

Price is not the target, but the problem motivation mentions spikes. This diagnostic compares price distributions after the tight label has been defined from load and wind alone. Many other factors affect pool price, so this is context rather than proof.

In [ ]:
test.groupby("tight").pool_price_cad_per_mwh.agg(
    ["count","mean","median","max"]
).rename(index={0:"Not tight",1:"Tight"})

In [ ]:
plt.figure(figsize=(8,5))
test.boxplot(column="pool_price_cad_per_mwh", by="tight", showfliers=False)
plt.suptitle("")
plt.title("Pool prices in held-out tight vs non-tight hours")
plt.xlabel("Tight class (0=no, 1=yes)")
plt.ylabel("CAD/MWh")
plt.show()

## 11. Automatic final summary

This cell prints the conclusions from the actual full dataset rather than hard-coding results from the small sample.

In [ ]:
base = results_df[results_df.model == "Same hour last week"].iloc[0]
best = results_df[results_df.model == best_name].iloc[0]

print("FINAL RESULTS")
print("="*72)
print(f"Initial tight rule: AIL >= {load_threshold:,.1f} MW and wind <= {wind_threshold:,.1f} MW.")
print(
    f"Weekly baseline: precision={base.precision:.3f}, recall={base.recall:.3f}, "
    f"F1={base.f1:.3f}, warnings={int(base.warnings_sent):,}."
)
print(
    f"Best ML ({best_name}): precision={best.precision:.3f}, recall={best.recall:.3f}, "
    f"F1={best.f1:.3f}, PR-AUC={best.pr_auc:.3f}, warnings={int(best.warnings_sent):,}."
)
print(f"F1 difference versus weekly baseline: {best.f1-base.f1:+.3f}.")
print(
    f"Changed rule: AIL >= {alt_load_threshold:,.1f} MW and "
    f"wind <= {alt_wind_threshold:,.1f} MW."
)
print(
    f"Warnings in the same test period changed from {original_warnings:,} "
    f"to {changed_warnings:,} ({changed_warnings-original_warnings:+,})."
)

## 12. Limitations and production extensions

With only timestamp, AIL, wind, and pool price, this is necessarily a historical warning benchmark. A real operational system should add forecast temperature, load forecasts, wind forecasts, outages/available generation, intertie conditions, and other supply information.

It should also choose its warning threshold using an explicit cost for **missed tight hours versus false alarms**, then use walk-forward retraining to adapt to changes in Alberta's generation fleet and demand.